<a href="https://colab.research.google.com/github/junseo25/ds2002-walmart-hurricane-project/blob/main/Weather_cleaning_for_Walmart_Midterm_scratch_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#copy past from previous labs and studios | From 2026-10-07 — APIs 101 and Ingestion with Retries — Studio


In [ ]:
import requests, pandas as pd
from urllib.parse import urlencode
from datetime import date, timedelta


BASE = 'https://archive-api.open-meteo.com/v1/archive'

END = date.today() - timedelta(days=3)     # let the archive settle
START = END - timedelta(days=2)            # a three-day window

params = {
    'latitude': 38.03, 'longitude': -78.51,          # Charlottesville
    'start_date': START.isoformat(),
    'end_date': END.isoformat(),
    'hourly': 'temperature_2m,precipitation',
    'timezone': 'America/New_York',
}

print(BASE + '?' + urlencode(params))



resp = requests.get(BASE, params=params, timeout=15)

print('status:', resp.status_code)

if resp.status_code == 200:
    data = resp.json()
    print('keys:', list(data.keys()))
    print('units:', data['hourly_units'])

    hourly = pd.DataFrame(data['hourly'])
    hourly['time'] = pd.to_datetime(hourly['time'])
    print(hourly.shape)
    print(hourly.head())

    wettest = hourly.loc[hourly['precipitation'].idxmax()]
    print('wettest hour:', wettest['time'], '|', wettest['precipitation'], 'mm')
else:
    print('Failed to retrieve data. Error details:', resp.json())

https://archive-api.open-meteo.com/v1/archive?latitude=38.03&longitude=-78.51&start_date=2026-10-28&end_date=2026-10-30&hourly=temperature_2m%2Cprecipitation&timezone=America%2FNew_York
status: 503
Failed to retrieve data. Error details: {'reason': 'The service is overloaded', 'error': True}


#Need this to be editable and malleable for other time zones and lat long coordinates based on data file


In [2]:
from pandas.io.api import read_csv
# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")


#Actually making the working code for the project


In [3]:
import requests
import pandas as pd
from urllib.parse import urlencode
from datetime import date, timedelta

#Since every store is in the same state, just set all state rows to the same value.
store['state'] = 'FL'

#Tallahassee is missing coords instead of dropping a store location
store.loc[store['city'] == 'Tallahassee', ['latitude', 'longitude']] = [30.4383, -84.2807]

#drop any duplicate rows that may have the same latitude and longitude
store = store.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)

#copied from lab
END = date.today() - timedelta(days=3)
START = END - timedelta(days=2)

BASE = 'https://archive-api.open-meteo.com/v1/archive'

#generates an empty list to inpute data into outside of the for loop so it doesnt get reset
all_weather_data = []
#iterates through all the rows so that the parameters are all correct
for i in range(len(store)):
    store_id = store.loc[i, 'store_id']
    city = store.loc[i, 'city']
    lat = store.loc[i, 'latitude']
    lon = store.loc[i, 'longitude']
    state = store.loc[i, 'state']

    # Assign appropriate timezone (Pensacola is in the Central Time Zone)
    tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'

#parameters that the API looks for, fixed by the if else statement above for tz
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': START.isoformat(),
        'end_date': END.isoformat(),
        'hourly': 'temperature_2m,precipitation',
        'timezone': tz,
    }

    response = requests.get(BASE, params=params)

#The response code is a successful one, no error occurs
    if response.status_code == 200:
        data = response.json()
        #Prints our that it can display the data for a specific store
        print(f"Successfully fetched data for {store_id} ({city}, {state})")
#finds hourly data for each store, if hourly data at a specific hour does not exist, it returns empty value
        hourly = data.get('hourly', {})
        print(hourly)
        if hourly:
            temp_stores = pd.DataFrame({
                'store_id': store_id,
                'city': city,
                'state': state,
                'time': hourly.get('time'),
                'temperature_2m': hourly.get('temperature_2m'),
                'precipitation': hourly.get('precipitation')
            })
#adds the data to the list
            all_weather_data.append(temp_stores)
    else:
        print(f"Failed to fetch data for {store_id} ({city}): Status {response.status_code}")





Successfully fetched data for FL-239 (Orlando, FL)
{'time': ['2026-10-28T00:00', '2026-10-28T01:00', '2026-10-28T02:00', '2026-10-28T03:00', '2026-10-28T04:00', '2026-10-28T05:00', '2026-10-28T06:00', '2026-10-28T07:00', '2026-10-28T08:00', '2026-10-28T09:00', '2026-10-28T10:00', '2026-10-28T11:00', '2026-10-28T12:00', '2026-10-28T13:00', '2026-10-28T14:00', '2026-10-28T15:00', '2026-10-28T16:00', '2026-10-28T17:00', '2026-10-28T18:00', '2026-10-28T19:00', '2026-10-28T20:00', '2026-10-28T21:00', '2026-10-28T22:00', '2026-10-28T23:00', '2026-10-29T00:00', '2026-10-29T01:00', '2026-10-29T02:00', '2026-10-29T03:00', '2026-10-29T04:00', '2026-10-29T05:00', '2026-10-29T06:00', '2026-10-29T07:00', '2026-10-29T08:00', '2026-10-29T09:00', '2026-10-29T10:00', '2026-10-29T11:00', '2026-10-29T12:00', '2026-10-29T13:00', '2026-10-29T14:00', '2026-10-29T15:00', '2026-10-29T16:00', '2026-10-29T17:00', '2026-10-29T18:00', '2026-10-29T19:00', '2026-10-29T20:00', '2026-10-29T21:00', '2026-10-29T22:00',

In [ ]:
if data:
    hourly = pd.DataFrame(data['hourly'])
else:                                  # keeps the notebook usable offline
    hourly = pd.DataFrame({
        'time': pd.date_range(START, periods=72, freq='h').astype(str),
        'temperature_2m': [12 + (i % 12) for i in range(72)],
        'precipitation': [0.0] * 30 + [1.4] * 6 + [0.0] * 36,
    })

hourly['time'] = pd.to_datetime(hourly['time'])

daily = (hourly.groupby(hourly['time'].dt.date)
         .agg(precipitation_total=('precipitation', 'sum'),
              temp_max=('temperature_2m', 'max'))
         .reset_index()
         .rename(columns={'time': 'date'}))
daily



# Combine into a master DataFrame
print(all_weather_data)
if all_weather_data:
    master_weather_df = pd.concat(all_weather_data, ignore_index=True)
    print(master_weather_df.head())

#PART 2: MAKING THE CODE WITH FUNCTIONS AFTER THE TESTING ABOVE

In [4]:
import requests, pandas as pd, numpy as np, requests, time
from urllib.parse import urlencode
from datetime import date, timedelta

# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")


#Since every store is in the same state, just set all state rows to the same value.
store['state'] = 'FL'

#Tallahassee is missing coords instead of dropping a store location
store.loc[store['city'] == 'Tallahassee', ['latitude', 'longitude']] = [30.4383, -84.2807]

#drop any duplicate rows that may have the same latitude and longitude
store = store.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)

#copied from lab
END = date.today() - timedelta(days=3)
START = END - timedelta(days=2)
LANDFALL = pd.Timestamp('2024-09-26')

BASE = 'https://archive-api.open-meteo.com/v1/archive'



#Creating function including the necessary parameters
def fetch_store_weather(lat, lon, tz, start_date, end_date, tries=3):
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': START,
        'end_date': END,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': tz,
    }
    #Error messages that are unacceptable and would cause errors
    RETRY_STATUSES = {429, 500, 502, 503, 504}
    #loop through the number of retry attempts
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame


START_DATE = '2024-09-01'
END_DATE = '2024-09-30'
all_store_weather = []

for i in range(len(store)):
    store_id = store.loc[i, 'store_id']
    city = store.loc[i, 'city']
    lat = store.loc[i, 'latitude']
    lon = store.loc[i, 'longitude']

    # Timezone alignment: Pensacola is Central, others Eastern
    tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'

    weather_df = fetch_store_weather(lat, lon, tz, START_DATE, END_DATE)

    if weather_df is not None:
        weather_df['store_id'] = store_id
        all_store_weather.append(weather_df)

master_weather = pd.concat(all_store_weather, ignore_index=True)
